In [56]:
import numpy as np
from iodata import load_one
from gbasis.wrappers import from_iodata
from gbasis.integrals.overlap import overlap_integral
from gbasis.evals.density import evaluate_density
from grid import GaussChebyshev, BeckeRTransform, MolGrid, BeckeWeights

In [3]:
mol = load_one("../examples/peptide.fchk")

In [70]:
# 1-electron reduced density matrix (1-RDM) from SCF calculation
rdm = mol.one_rdms.get("scf")


In [35]:
coeffs = mol.mo.coeffs
occs = mol.mo.occs
rdm_raw = np.dot(coeffs * occs, coeffs.T)

In [69]:
mo = mol.mo
# Spin-resolved density matrices
# Alpha
rdm_a = (mo.coeffsa * mo.occsa) @ mo.coeffsa.conj().T

# Beta
rdm_b = (mo.coeffsb * mo.occsb) @ mo.coeffsb.conj().T

# Total density
rdm_total = rdm_a + rdm_b

# Verify consistency
rdm_raw = (mo.coeffs * mo.occs) @ mo.coeffs.conj().T

assert np.allclose(rdm_total, rdm_raw)
assert np.allclose(rdm_total, rdm)

In [71]:
np.allclose(rdm_raw, rdm)

True

In [ ]:
# basis function from mol object
basis = from_iodata(mol)

In [73]:
# basis function from mol object
basis = from_iodata(mol)

# AO overlap matrix
S = overlap_integral(basis)

# Number of electrons considering non-orthogonal basis
n_alpha = np.trace(rdm_a @ S)
n_beta = np.trace(rdm_b @ S)

print("Alpha electrons:", n_alpha)
print("Beta electrons:", n_beta)

# Check AO orthonormality
print(np.allclose(mo.coeffsa.T @ S @ mo.coeffsa,
                  np.eye(mo.coeffsa.shape[1])))

Alpha electrons: 19.99999998368834
Beta electrons: 19.99999998364403
False


In [74]:
C = mo.coeffsa
occupied = mo.occsa > 1e-8

C_occ = C[:, occupied]
C_vir = C[:, ~occupied]

S_oo = C_occ.T @ S @ C_occ
S_vv = C_vir.T @ S @ C_vir
S_ov = C_occ.T @ S @ C_vir

print("Occupied-occupied:",
      np.max(np.abs(S_oo - np.eye(S_oo.shape[0]))))

print("Virtual-virtual:",
      np.max(np.abs(S_vv - np.eye(S_vv.shape[0]))))

print("Occupied-virtual:",
      np.max(np.abs(S_ov)))

Occupied-occupied: 8.716500951577721e-09
Virtual-virtual: 6.699490752103116e-06
Occupied-virtual: 9.97960157988089e-07


In [75]:
# Construct molecular grid
oned = GaussChebyshev(npoints=100)
rgrid = BeckeRTransform(0.0, R=1.5).transform_1d_grid(oned)

grid = MolGrid.from_preset(
    atnums=mol.atnums,
    atcoords=mol.atcoords,
    rgrid=rgrid,
    preset="fine",
    aim_weights=BeckeWeights(),
    store=True,
)

# Evaluate spin-resolved densities
dens_a = evaluate_density(rdm_a, basis, grid.points)
dens_b = evaluate_density(rdm_b, basis, grid.points)

# Total electron density
moldens_raw = dens_a + dens_b
moldens = evaluate_density(rdm, basis, grid.points)

# Spin density
spin_dens = dens_a - dens_b

C:\Users\ccast\modules\grid\src\grid\atomgrid.py:907: UserWarning: Lebedev weights are negative which can introduce round-off errors.
  sphere_grid = AngularGrid(degree=deg_i, method=method)


In [76]:
print("Alpha electrons:", grid.integrate(dens_a))
print("Beta electrons:", grid.integrate(dens_b))
print("Total electrons:", grid.integrate(moldens))
print("Total electrons (raw):", grid.integrate(moldens_raw))
print("Spin polarization:", grid.integrate(spin_dens))

Alpha electrons: 20.0001522877686
Beta electrons: 20.000152287724305
Total electrons: 40.000304584947834
Total electrons (raw): 40.000304575492905
Spin polarization: 4.432237930984075e-11


In [77]:
print("MO type:", mo.kind)
print("Maximum spin density:", np.max(spin_dens))
print("Minimum spin density:", np.min(spin_dens))
print("Integrated absolute spin density:",
      grid.integrate(np.abs(spin_dens)))

MO type: unrestricted
Maximum spin density: 2.268368826108258e-11
Minimum spin density: -4.860112312599085e-11
Integrated absolute spin density: 6.542090526589624e-10


In [78]:
spin_dens_true = mol.one_rdms.get("scf_spin")
spin_dens_true = evaluate_density(spin_dens_true, basis, grid.points)
print("Integrated absolute spin density:",
      grid.integrate(np.abs(spin_dens_true)))

Integrated absolute spin density: 4.2009521714510993e-13
